In [1]:
from git_utils import github_blob_to_raw, get_github_remote_file_size, github_to_api_contents, github_blob_to_raw, github_to_api_contents
from utils import create_directory, run, create_index_db, csv_to_list_of_dicts

from dsi.dsi import DSI

In [17]:
repo_path = "ssh://git@lisdi-git.lanl.gov:10022/personal/test_dsi_federated.gitt"

In [18]:
tmp_folder = "xyzr4"

In [19]:
# clone the repo
run(f"git clone {repo_path} {tmp_folder}")

RuntimeError: Command failed: git clone ssh://git@lisdi-git.lanl.gov:10022/personal/test_dsi_federated.gitt xyzr4
stdout:

stderr:
Cloning into 'xyzr4'...
remote: 
remote: ========================================================================
remote: 
remote: ERROR: The project you were looking for could not be found or you don't have permission to view it.

remote: 
remote: ========================================================================
remote: 
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.


In [5]:
# Get the list of databases 
catalogue_index = f"{tmp_folder}/index_db.csv"
catalogue_list = csv_to_list_of_dicts(catalogue_index)

In [ ]:
# Pull data from each

In [7]:
for db in catalogue_list:
    location = db['location']
    print(location)

github
ch-fe.lanl.gov
darwin.lanl.gov


In [11]:
import uuid
s = uuid.uuid4().hex[:8]
print(s)

17f27cc5


In [12]:
tmp_folder = f"_tmp_{uuid.uuid4().hex[:8]}"

In [13]:
tmp_folder

'_tmp_2e172e84'

In [20]:
folder = "_tmp_37ed6066"

In [21]:
def get_github_remote_file_size(url: str, timeout: int = 20, github_token: str | None = None) -> int:
    """
    Get the size of a remote file on GitHub in bytes without downloading it.
    Tries multiple methods for robustness:

    Args:
        url: The GitHub URL of the file (can be a blob/tree URL or raw URL)
        timeout: Timeout in seconds for network requests
        github_token: Optional GitHub token for authenticated requests (can help with rate limits)

    Returns:
        The size of the remote file in bytes.
    """
    # 1) Prefer GitHub API if it looks like a GitHub blob/tree link
    api_url = github_to_api_contents(url)
    if api_url:
        headers = {"Accept": "application/vnd.github+json"}
        if github_token:
            headers["Authorization"] = f"Bearer {github_token}"
        r = requests.get(api_url, headers=headers, timeout=timeout)
        if r.ok:
            js = r.json()
            if isinstance(js, dict) and js.get("type") == "file" and "size" in js:
                return int(js["size"])
        # If API fails, try raw URL next

    # 2) If it's a GitHub blob URL, switch to raw for HTTP size checks
    raw_url = github_blob_to_raw(url) or url

    # 3) HEAD
    r = requests.head(raw_url, allow_redirects=True, timeout=timeout)
    cl = r.headers.get("Content-Length")
    if r.ok and cl and cl.isdigit():
        return int(cl)

    # 4) Range GET fallback
    r = requests.get(raw_url, headers={"Range": "bytes=0-0"}, allow_redirects=True, timeout=timeout)
    cr = (r.headers.get("Content-Range") or "").strip()
    if "/" in cr:
        total = cr.split("/")[-1].strip()
        if total.isdigit():
            return int(total)

    cl = r.headers.get("Content-Length")
    if cl and cl.isdigit():
        return int(cl)

    return 0

In [23]:
import requests

In [25]:
fs = get_github_remote_file_size("https://github.com/lanl/dsi/tree/ai_dev/tools/ai_search/data/oceans_11/ocean_11_datasets.db")

In [26]:
fs

20480